# 01 · Download the 10-Q

Stage 1 of the indexing pipeline (see `ARCHITECTURE.md`).

Downloads Ares Commercial Real Estate's (ACRE) 10-Q for the quarter ended June 30, 2026 from SEC EDGAR and saves it to `data/raw/`.

SEC rules: every request must send a `User-Agent` with your name and email, and stay under 10 requests per second.

In [2]:
import os
from pathlib import Path

import requests
from dotenv import load_dotenv

load_dotenv("../.env")

FILING_URL = "https://www.sec.gov/Archives/edgar/data/1529377/000162828026052072/acre-20260630.htm"
RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = RAW_DIR / "acre-20260630.htm"

USER_AGENT = os.getenv("SEC_USER_AGENT")
assert USER_AGENT, "SEC_USER_AGENT is missing from .env"

In [3]:
if OUT_PATH.exists():
    print("Already downloaded:", OUT_PATH)
else:
    resp = requests.get(
        FILING_URL,
        headers={"User-Agent": USER_AGENT, "Accept-Encoding": "gzip, deflate"},
        timeout=30,
    )
    resp.raise_for_status()
    OUT_PATH.write_bytes(resp.content)
    print("Saved:", OUT_PATH)

print(f"Size: {OUT_PATH.stat().st_size / 1_000_000:.1f} MB")

Already downloaded: ..\data\raw\acre-20260630.htm
Size: 1.6 MB


## Quick look

Check that it's the real filing, not an SEC error page.

In [ ]:
import warnings

from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning

# SEC filings are inline XBRL (XHTML); parsing them as HTML is intended.
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

soup = BeautifulSoup(OUT_PATH.read_bytes(), "lxml")

# Drop the hidden XBRL metadata block so only the visible filing text remains.
for hidden in soup.find_all("ix:header"):
    hidden.decompose()

text = soup.get_text(" ", strip=True)

print("Title:", soup.title.string if soup.title else "(none)")
print("Tables:", len(soup.find_all("table")))
print("Words:", f"{len(text.split()):,}")
print()
print(text[:600])

Title: acre-20260630
Tables: 48
Words: 31,833

acre-20260630 UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 ___________________________________________________________________ FORM 10-Q ☒ QUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the quarterly period ended June 30, 2026 OR ☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the transition period from _____ to _____ Commission File No. 001-35517 ARES COMMERCIAL REAL ESTATE CORPORATION (Exact name of Registrant as specified in its charter) Maryland 45-3148087 (State or other 


: 